In [ ]:
import segmentation_models_pytorch as smp

from sklearn.model_selection import train_test_split 

import os, time, numpy as np
from typing import Dict
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split


# Constants

In [3]:
DEVICE  = torch.device(f"cuda:0" if torch.cuda.is_available() else "cpu")
#DEVICE  = "cpu"
print(DEVICE)


cuda:0


In [4]:
NUM_CLASSES = 10
EPOCHS = 25


In [5]:
base_path = "../Dataset"

dataset = base_path + "/s2-utm-33N-18E-242N-2018"
train_geojson_path = base_path + "/br-18E-242N-crop-labels-train-2018.geojson"

folder = "/DS3"
subfolder1 = "/Temporal Data"

data_path = dataset+folder
temporal_data_path = dataset+folder+subfolder1

classification_model_path = "../Classification models/Spatial"
temporal_classification_model_path = "../Classification models/Temporal"


In [ ]:
# -----------------------
# Dataset (temporal)
# -----------------------
class TemporalSegDataset(Dataset):
    def __init__(self, X: np.ndarray, Y: np.ndarray, T):
        """
        X: (N,T,4,H,W), Y: (N,H,W)
        """
        assert X.ndim == 5 and Y.ndim == 3, "X=(N,T,4,H,W), Y=(N,H,W)"
        assert X.shape[0] == Y.shape[0], "X, Y must share N"
        assert X.shape[1] == T, f"T mismatch: X has {X.shape[1]}, expected {T}"
        self.X = X
        self.Y = Y

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx]).float()   # (T,4,H,W)
        y = torch.from_numpy(self.Y[idx]).long()    # (H,W)
        return x, y

def make_loaders_temporal(trainX, trainY, T, batch, val_ratio=0.2, workers=4, seed=42):
    X = np.load(trainX)  # (N,T,4,H,W)
    Y = np.load(trainY)  # (N,H,W)
    ds = TemporalSegDataset(X, Y, T)
    n_val = int(len(ds)*val_ratio); n_tr = len(ds)-n_val
    g = torch.Generator().manual_seed(seed)
    tr, va = random_split(ds, [n_tr, n_val], generator=g)
    tr_ld = DataLoader(tr, batch_size=batch, shuffle=True, pin_memory=True, num_workers=workers)
    va_ld = DataLoader(va, batch_size=batch, shuffle=False, pin_memory=True, num_workers=workers)
    return tr_ld, va_ld

def make_loader_test(testX, testY, T, batch, workers=4):
    X = np.load(testX); Y = np.load(testY)
    ds = TemporalSegDataset(X, Y, T=T)
    return DataLoader(ds, batch_size=batch, shuffle=False, pin_memory=True, num_workers=workers)

# -----------------------
# Temporal models
# -----------------------
class EarlyFusionSeg(nn.Module):
    """(B,T,4,H,W) -> reshape to (B,4T,H,W) -> DeepLabV3+"""
    def __init__(self, num_classes: int, backbone: str, T: int):
        super().__init__()
        self.T = T
        self.net = smp.DeepLabV3Plus(
            encoder_name=backbone, encoder_weights=None,
            in_channels=4*T, classes=num_classes
        )
    def forward(self, x):  # (B,T,4,H,W)
        B,T,C,H,W = x.shape
        if T != self.T: raise ValueError(f"Expected T={self.T}, got {T}")
        x = x.reshape(B, T*C, H, W)      # (B,4T,H,W)
        return self.net(x)

class EarlyFusionUNet(nn.Module):
    """(B,T,4,H,W) → (B,4T,H,W) → U-Net"""
    def __init__(self, num_classes: int, backbone: str, T: int):
        super().__init__()
        self.T = T
        self.net = smp.Unet(
            encoder_name=backbone, encoder_weights=None,
            in_channels=4*T, classes=num_classes
        )
    def forward(self, x):  # x: (B,T,4,H,W)
        B,T,C,H,W = x.shape
        if T != self.T: raise ValueError(f"Expected T={self.T}, got {T}")
        x = x.reshape(B, T*C, H, W)     # (B,4T,H,W)
        return self.net(x)


# class Temporal3DStem(nn.Module):
#     """(B,T,4,H,W) -> Conv3D -> temporal pool -> (B,C2D,H,W)"""
#     def __init__(self, c2d=64):
#         super().__init__()
#         self.stem = nn.Sequential(
#             nn.Conv3d(4, 32, (3,3,3), padding=1, bias=False),
#             nn.BatchNorm3d(32), nn.ReLU(inplace=True),
#             nn.Conv3d(32, c2d, (3,3,3), padding=1, bias=False),
#             nn.BatchNorm3d(c2d), nn.ReLU(inplace=True),
#         )
#     def forward(self, x):  # (B,T,4,H,W)
#         x = x.permute(0,2,1,3,4)        # (B,4,T,H,W)
#         f = self.stem(x)                # (B,C2D,T,H,W)
#         f = f.mean(dim=2)               # temporal avg → (B,C2D,H,W)
#         return f

# class Temporal3D_DeepLab(nn.Module):
#     def __init__(self, num_classes: int, backbone: str, c2d=64):
#         super().__init__()
#         self.temporal = Temporal3DStem(c2d)
#         self.head = smp.DeepLabV3Plus(
#             encoder_name=backbone, encoder_weights=None,
#             in_channels=c2d, classes=num_classes
#         )
#     def forward(self, x):               # (B,T,4,H,W)
#         f = self.temporal(x)            # (B,C2D,H,W)
#         return self.head(f)

# class ConvLSTMCell(nn.Module):
#     def __init__(self, in_ch, hidden_ch, k=3):
#         super().__init__()
#         p = k//2
#         self.conv = nn.Conv2d(in_ch+hidden_ch, 4*hidden_ch, k, padding=p, bias=True)
#         self.hidden_ch = hidden_ch
#     def forward(self, x, state):
#         h, c = state
#         z = torch.cat([x, h], dim=1)
#         gates = self.conv(z)
#         i,f,o,g = gates.chunk(4, dim=1)
#         i = torch.sigmoid(i); f = torch.sigmoid(f); o = torch.sigmoid(o); g = torch.tanh(g)
#         c = f*c + i*g
#         h = o*torch.tanh(c)
#         return h, c
#     def init_state(self, B, H, W, device):
#         h = torch.zeros(B, self.hidden_ch, H, W, device=device)
#         c = torch.zeros(B, self.hidden_ch, H, W, device=device)
#         return h, c

# class ConvLSTM_DeepLab(nn.Module):
#     """ConvLSTM over T, then 2D DeepLabV3+ head."""
#     def __init__(self, num_classes: int, backbone: str, hidden=64):
#         super().__init__()
#         self.embed = nn.Conv2d(4, 32, 3, padding=1)
#         self.rnn   = ConvLSTMCell(32, hidden, k=3)
#         self.head  = smp.DeepLabV3Plus(
#             encoder_name=backbone, encoder_weights=None,
#             in_channels=hidden, classes=num_classes
#         )
#     def forward(self, x):  # (B,T,4,H,W)
#         B,T,C,H,W = x.shape
#         h,c = self.rnn.init_state(B,H,W, device=x.device)
#         for t in range(T):
#             xt = torch.relu(self.embed(x[:,t]))   # (B,32,H,W)
#             h,c = self.rnn(xt, (h,c))             # (B,Hid,H,W)
#         return self.head(h)

# def build_model(kind: str, num_classes: int, backbone: str, T: int) -> nn.Module:
#     if kind == "early":
#         return EarlyFusionSeg(num_classes, backbone, T)
#     if kind == "3dstem":
#         return Temporal3D_DeepLab(num_classes, backbone, c2d=64)
#     if kind == "convlstm":
#         return ConvLSTM_DeepLab(num_classes, backbone, hidden=64)
#     raise ValueError(kind)

# -----------------------
# Loss & metrics
# -----------------------
class DiceLossMulti(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__(); self.eps=eps
    def forward(self, logits, target):
        probs = torch.softmax(logits, dim=1)
        B,K,H,W = probs.shape
        tgt1h = torch.zeros_like(probs).scatter_(1, target.unsqueeze(1), 1)
        inter = (probs*tgt1h).sum((0,2,3))
        union = probs.sum((0,2,3)) + tgt1h.sum((0,2,3))
        dice = (2*inter + self.eps) / (union + self.eps)
        return 1 - dice.mean()

def loss_fn(logits, y, ce_weight=None):
    ce = torch.nn.CrossEntropyLoss(weight=ce_weight)(logits, y)
    dice = DiceLossMulti()(logits, y)
    return 0.5*ce + 0.5*dice

@torch.no_grad()
def confusion_matrix(preds, target, num_classes, ignore_index=None):
    if ignore_index is not None:
        mask = target != ignore_index
        preds = preds[mask]; target = target[mask]
    k = num_classes*target.view(-1) + preds.view(-1)
    return torch.bincount(k, minlength=num_classes**2).reshape(num_classes, num_classes)

@torch.no_grad()
def metrics_from_conf(conf: torch.Tensor) -> Dict[str,float]:
    tp = torch.diag(conf).float()
    gt = conf.sum(1).float(); pd = conf.sum(0).float()
    union = gt + pd - tp
    iou = torch.where(union>0, tp/union.clamp(min=1), torch.zeros_like(tp))
    miou = iou.mean().item()
    pixacc = (tp.sum()/conf.sum().clamp(min=1)).item()
    dice = torch.where((gt+pd)>0, 2*tp/(gt+pd).clamp(min=1), torch.zeros_like(tp))
    return {"mIoU": miou, "pixel_acc": pixacc, "mF1": dice.mean().item(),
            "IoU_per_class": iou.cpu().tolist(), "F1_per_class": dice.cpu().tolist(),
            "support_per_class": gt.cpu().tolist()}

# -----------------------
# Train / Eval
# -----------------------
def train_epoch(model, loader, opt, scaler, device, ce_weight=None):
    model.train()
    tot, tsum = 0, 0.0
    
    for x,y in loader:
        x = x.to(DEVICE).float()      # [B, T, 4, 64, 64]
        y = y.to(DEVICE).long()  
        
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits = model(x)
            loss = loss_fn(logits, y, ce_weight)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        bs = x.size(0); tot += bs; tsum += loss.item()*bs
    return tsum/max(tot,1)

@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    model.eval()
    conf = torch.zeros((num_classes,num_classes), device=device, dtype=torch.int64)
    tot, tsum = 0, 0.0
    
    for x,y in loader:
        x = x.to(DEVICE).float()      # [B, 4, 64, 64]
        y = y.to(DEVICE).long()  
        with torch.amp.autocast('cuda'):
            logits = model(x); loss = loss_fn(logits, y)
        preds = logits.argmax(1)
        conf += confusion_matrix(preds, y, num_classes)
        bs = x.size(0); tot += bs; tsum += loss.item()*bs
    mets = metrics_from_conf(conf); mets["loss"] = tsum/max(tot,1)
    return mets, conf


In [7]:
class ImageLabelDataset(Dataset):
    def __init__(self, image_array, label_array):
        self.images = torch.tensor(image_array, dtype=torch.float32)  # (N, 4, 64, 64)
        self.labels = torch.tensor(label_array, dtype=torch.float32)  # (N, 1, 64, 64)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]



In [ ]:
# Data
final_gan_data = np.load((temporal_data_path+'/14_final_TGAN_data.npy'))
final_gan_labels = np.load((temporal_data_path+'/15_final_TGAN_labels.npy'))
doy_list = np.load((temporal_data_path+'/13_selected_dates_DOY_normalised.npy'))

labels_upadated = np.where(final_gan_labels == -1, 0, final_gan_labels)
train_data, validation_data, train_labels, validation_labels = train_test_split(final_gan_data, labels_upadated, test_size=0.4, random_state= 2) 

image_data = np.transpose(train_data, (0, 1, 4, 2, 3)) 
# reshape to (B,T,C,H,W) for early-fusion model

dataset = ImageLabelDataset(image_data, train_labels)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

validation_image_data = np.transpose(validation_data, (0, 1, 4, 2, 3)) 
# reshape to (B,T,C,H,W) for early-fusion model

val_dataset = ImageLabelDataset(validation_image_data, validation_labels)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True)


In [28]:
SEED =2
EPOCHS = 25
T = 27

model_name_list = ['unet_r50', 'deeplab_r50', 'unet_r101', 'deeplab_r101']
selected_model_index = 0
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)


# Model
# model = EarlyFusionSeg(10, encoder_name, T).to(DEVICE)
model = EarlyFusionUNet(num_classes=10, backbone=encoder_name, T=T).to(DEVICE)


# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

# Train with early stop on val mIoU
best, bad, patience = -1.0, 0, 2
# os.makedirs(args.out, exist_ok=True)
# best_path = os.path.join(args.out, "best.pt")
t0 = time.time()
for ep in range(1, EPOCHS+1):
    tr_loss = train_epoch(model, loader, opt, scaler, DEVICE)
    val_mets, _ = evaluate(model, val_loader, DEVICE, 10)
    sched.step()
    print(f"[{ep:03d}/{EPOCHS}] tr_loss={tr_loss:.4f} | "
            f"val_loss={val_mets['loss']:.4f} | val_mIoU={val_mets['mIoU']:.4f} "
            f"| val_acc={val_mets['pixel_acc']:.4f}")

    if val_mets["mIoU"] > best:
        best = val_mets["mIoU"]; bad = 0
        #torch.save({"model": model.state_dict(), "epoch": ep, "best": best}, best_path)
    else:
        bad += 1
        if bad >= patience:
            print("Early stopping."); break

print(f"Finished in {(time.time()-t0)/60:.1f} min. Best val mIoU={best:.4f}")





unet_r50 resnet50


C:\Users\Sai Suhaas\AppData\Local\Temp\ipykernel_31864\1036336709.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
C:\Users\Sai Suhaas\AppData\Local\Temp\ipykernel_31864\40402494.py:200: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\Sai Suhaas\AppData\Local\Temp\ipykernel_31864\40402494.py:217: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[001/25] tr_loss=0.9406 | val_loss=0.5298 | val_mIoU=0.7155 | val_acc=0.8219
[002/25] tr_loss=0.5110 | val_loss=0.3673 | val_mIoU=0.7901 | val_acc=0.8712
[003/25] tr_loss=0.4042 | val_loss=0.3213 | val_mIoU=0.8034 | val_acc=0.8786
[004/25] tr_loss=0.3424 | val_loss=0.2879 | val_mIoU=0.8259 | val_acc=0.8930
[005/25] tr_loss=0.3066 | val_loss=0.2467 | val_mIoU=0.8515 | val_acc=0.9108
[006/25] tr_loss=0.2783 | val_loss=0.2495 | val_mIoU=0.8474 | val_acc=0.9084
[007/25] tr_loss=0.2623 | val_loss=0.2324 | val_mIoU=0.8590 | val_acc=0.9153
[008/25] tr_loss=0.2447 | val_loss=0.2130 | val_mIoU=0.8702 | val_acc=0.9224
[009/25] tr_loss=0.2296 | val_loss=0.2133 | val_mIoU=0.8682 | val_acc=0.9210
[010/25] tr_loss=0.2213 | val_loss=0.1969 | val_mIoU=0.8769 | val_acc=0.9263
[011/25] tr_loss=0.2118 | val_loss=0.1870 | val_mIoU=0.8859 | val_acc=0.9319
[012/25] tr_loss=0.2025 | val_loss=0.2009 | val_mIoU=0.8794 | val_acc=0.9286
[013/25] tr_loss=0.1899 | val_loss=0.1810 | val_mIoU=0.8857 | val_acc=0.9321

In [ ]:
# test_loader = make_loader_test(args.testX, args.testY, T, batch=16)

# Test
# ck = torch.load(best_path, map_location=DEVICE)
# model.load_state_dict(ck["model"])
# test_mets, test_conf = evaluate(model, test_loader, DEVICE, 10)
# print(f"[TEST] loss={test_mets['loss']:.4f} | acc={test_mets['pixel_acc']:.4f} "
#         f"| mIoU={test_mets['mIoU']:.4f} | mF1={test_mets['mF1']:.4f}")
# print("IoU_per_class: " + ",".join(f"{x:.4f}" for x in test_mets["IoU_per_class"]) + "\n")
# print("F1_per_class: "  + ",".join(f"{x:.4f}" for x in test_mets["F1_per_class"])  + "\n")

# Save confusion matrix and per-class metrics
# np.save(os.path.join(args.out, "test_confusion.npy"), test_conf.cpu().numpy())
# with open(os.path.join(args.out, "test_metrics.txt"), "w") as f:
#     for k,v in test_mets.items():
#         if isinstance(v, (list,tuple)): continue
#         f.write(f"{k}: {v}\n")
#     f.write("IoU_per_class: " + ",".join(f"{x:.4f}" for x in test_mets["IoU_per_class"]) + "\n")
#     f.write("F1_per_class: "  + ",".join(f"{x:.4f}" for x in test_mets["F1_per_class"])  + "\n")
